# Bloomberg — Senior ML Research Engineer (Sustainable Finance)
## First-round interview preparation | Python • ML research • NLP • ESG

**How to use:** Read the explanation, attempt each exercise **before** viewing the worked solution, execute the code, and practise explaining complexity, trade-offs, and production considerations aloud. All datasets are **synthetic** and no Bloomberg proprietary data is needed.

**Important:** The HackerRank URL is an interview link; it does **not** establish that this is a timed online assessment. Some past Bloomberg interviews have included live coding using HackerRank, but format varies.

**Preparation roadmap:** 1) coding and data structures; 2) ML/statistics; 3) sustainable-finance research problems; 4) NLP and LLMs; 5) research-to-production system design; 6) verbal mock interviews.

## 0. Role mapping and interview strategy

Published responsibilities include real-time news controversy detection, estimation of GHG emissions for listed companies, materiality assessment, and explainability. Expect potentially different rounds: algorithmic coding, ML research scenarios, technical discussion of past work, and applied system design. **Nothing below represents leaked or guaranteed questions.**

For every applied question use **P-D-M-E-P**: **Problem** and decisions → **Data** and labels → **Model** and baselines → **Evaluation** including leakage → **Production** including monitoring and human review.

At senior level discuss nonstationarity, entity resolution, quality of labels, coverage bias, publication timestamps, interpretability, and controls.

**Interview response formula:** clarify assumptions → simple baseline → justify a stronger approach → give evaluation protocol → describe failures/monitoring.

In [4]:
import sys, math, re, heapq, random
from collections import Counter, defaultdict, deque
import numpy as np
import pandas as pd
print('Python:', sys.version.split()[0], '| NumPy:', np.__version__, '| pandas:', pd.__version__)
SEED = 42
rng = np.random.default_rng(SEED)

Python: 3.11.13 | NumPy: 2.4.6 | pandas: 3.0.3


## 1. Python fundamentals and algorithmic complexity

**`dict`**: key → value lookups, average O(1). **`defaultdict(list)`**: automatically initializes a missing list, useful when grouping news by issuer. **`Counter`**: counts occurrences; `most_common(k)` returns frequently occurring items. **`deque`**: O(1) append/pop at either end; a Python list's `pop(0)` costs O(n). **`heapq`**: priority queue; maintain top *k* entries in O(n log k).

For each coding question articulate input assumptions, time complexity, space complexity, and edge cases.

In [5]:
events = [('ABC','climate'),('XYZ','governance'),('ABC','climate'),('ABC','labor')]


In [6]:
events = [('ABC','climate'),('XYZ','governance'),('ABC','climate'),('ABC','labor')]
by_issuer = defaultdict(list)
for issuer, topic in events:
    by_issuer[issuer].append(topic)
print(dict(by_issuer))


{'ABC': ['climate', 'climate', 'labor'], 'XYZ': ['governance']}


In [ ]:
(topic for _, topic in events)

['climate', 'governance', 'climate', 'labor']

In [ ]:
print('Event counts:', Counter(topic for _, topic in events))

print('Sorted top two:', Counter(topic for _, topic in events).most_common(2))

### Exercise 1 — Sliding-window controversy alerts
**Question:** Given an ascending list of event times in integer minutes, find the largest number of events within any inclusive interval of `window` minutes.

Example: `[0, 5, 8, 30, 31]`, `window=10` → `3`.

**Think aloud:** Is input sorted? Are times inclusive? Can there be duplicate timestamps? What if `window=0`?

**Expected:** two pointers, time O(n) once sorted, auxiliary space O(1).

In [ ]:
# Your attempt
def max_events_in_window_practice(times, window):
    # TODO: solve before reviewing the answer
    raise NotImplementedError

# Uncomment to test after implementing:
# assert max_events_in_window_practice([0, 5, 8, 30, 31], 10) == 3

<details><summary><strong>Worked solution + explanation</strong></summary>
Maintain a left pointer. For each right endpoint, shrink the interval until `times[right] - times[left] <= window`. Each pointer advances at most *n* times: O(n).
</details>

In [ ]:
def max_events_in_window(times, window):
    if window < 0: raise ValueError('window must be non-negative')
    left = best = 0
    for right, current in enumerate(times):
        while current - times[left] > window:
            left += 1
        best = max(best, right - left + 1)
    return best

assert max_events_in_window([0,5,8,30,31],10)==3
assert max_events_in_window([],10)==0
assert max_events_in_window([4,4,5],0)==2
print('Passed sliding-window tests')

### Exercise 2 — Top-k ESG categories (hash maps + heaps)
Design a function taking `[('issuer','topic'), ...]` and integer *k*, and returning the top-k topics and their counts. Why might a heap be better than sorting all unique topics?

In [ ]:
def top_k_topics(records, k):
    if k <= 0: return []
    counts = Counter(topic for _,topic in records)
    return heapq.nlargest(k, counts.items(), key=lambda item:(item[1], item[0]))

print(top_k_topics(events, 2))
# Complexity: count O(n), heap selection O(m log k) for m unique categories.

### Exercise 3 — Supplier network (graphs, BFS)
A controversy at a supplier may propagate to customers through the supply chain. Find all companies reachable from one issuer in a *directed* graph. Discuss whether direction represents supplier→buyer or buyer→supplier before implementing.

**Complexity:** O(V+E) for reachable graph traversal.

In [ ]:
supply_chain = {'A':['B','C'], 'B':['D'], 'C':['D','E'], 'D':[], 'E':['A']}
def bfs_reachable(graph, start):
    seen = {start}; q = deque([start]); visit_order = []
    while q:
        node = q.popleft(); visit_order.append(node)
        for neighbour in graph.get(node, []):
            if neighbour not in seen:
                seen.add(neighbour); q.append(neighbour)
    return visit_order

assert set(bfs_reachable(supply_chain, 'A')) == {'A','B','C','D','E'}
print(bfs_reachable(supply_chain,'A'))

### Exercise 4 — Stream de-duplication
An issuer may be referenced in multiple near-duplicate news articles. Write a stable de-duplicator using `(issuer_id, normalized_headline)` as the key.

**Discussion:** Exact duplicate removal is not semantic deduplication; syndicated stories, language variants and edited headlines require clustering/embeddings and provenance.

In [ ]:
articles = [
    {'issuer':'A','headline':'Plant closes after spill'},
    {'issuer':'A','headline':' plant closes after spill '},
    {'issuer':'B','headline':'Plant closes after spill'},
]
def dedupe_articles(rows):
    seen = set(); result = []
    for row in rows:
        key = (row['issuer'], ' '.join(row['headline'].lower().split()))
        if key not in seen:
            seen.add(key); result.append(row)
    return result
assert len(dedupe_articles(articles)) == 2
print(dedupe_articles(articles))

### Exercise 5 — Grouped rolling counts with pandas
Count the last 7 calendar days of synthetic articles per issuer (including the current day). Interview trap: a **7-row** rolling window is not necessarily **7 days**.

In [ ]:
news = pd.DataFrame({
    'issuer':['A','A','A','B','B','B'],
    'published_at':pd.to_datetime(['2026-01-01','2026-01-03','2026-01-11','2026-01-01','2026-01-02','2026-01-15']),
    'severity':[2,4,3,1,5,2]})
def rolling_article_counts(frame, days=7):
    parts=[]
    for issuer, g in frame.groupby('issuer'):
        g=g.sort_values('published_at').copy()
        g['rolling_count'] = (g.set_index('published_at')['severity']
                              .rolling(f'{days}D', closed='both').count().to_numpy().astype(int))
        parts.append(g)
    return pd.concat(parts).sort_values(['issuer','published_at']).reset_index(drop=True)
print(rolling_article_counts(news))

## 2. Statistics and ML research foundations

**Review questions (answer aloud):**
1. *Precision vs recall?* In alert triage, higher precision reduces analyst workload; higher recall misses fewer severe cases. Discuss operating threshold and costs.
2. *F1 vs PR-AUC?* F1 evaluates one threshold; PR-AUC / average precision summarizes ranking across thresholds (definition/interpolation differs). For rare controversies, PR curves are often more informative than ROC-AUC.
3. *Type I/II errors?* False alarm vs missed true event (with null defined appropriately).
4. *Confidence interval vs prediction interval?* Uncertainty in estimated mean vs range for a new observation.
5. *What is leakage?* Features measured **after** prediction time, duplicates crossing folds, target-derived proxies, post-event labels.
6. *Why temporal validation?* Mimics deployment on future data; random split can overstate performance due to repeated issuers/events and changing regimes.
7. *Calibration?* Among predictions near 0.7, roughly 70% should be positive in a well-calibrated model on the evaluation population.

### Worked ML study — Highly imbalanced controversy classification
Use a synthetic population with a low positive rate, no external downloads. Compare dummy baseline and logistic regression, and tune a decision threshold **on validation only**. Report held-out test metrics without fitting on test labels.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (average_precision_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix, precision_recall_curve)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
X,y=make_classification(n_samples=3000, n_features=12, n_informative=5,
                        n_redundant=2, weights=[0.95,0.05], flip_y=0.015, random_state=42)
# Toy demonstration: chronological indices stand in for arrival ordering; real data needs genuine timestamps.
Xtr, Xva, Xte = X[:1800],X[1800:2400],X[2400:]
ytr,yva,yte = y[:1800],y[1800:2400],y[2400:]
model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000))
model.fit(Xtr,ytr)
val_probs=model.predict_proba(Xva)[:,1]
thresholds=np.linspace(0.05,0.95,91)
val_f1=[]
for t in thresholds:
    _,_,f,_=precision_recall_fscore_support(yva,val_probs>=t,average='binary',zero_division=0)
    val_f1.append(f)
best_t=float(thresholds[int(np.argmax(val_f1))])
test_probs=model.predict_proba(Xte)[:,1]
precision, recall, f1,_=precision_recall_fscore_support(yte,test_probs>=best_t,average='binary',zero_division=0)
print({'threshold_from_validation':round(best_t,2),'precision':round(precision,3),
       'recall':round(recall,3),'f1':round(f1,3),'test_AP':round(average_precision_score(yte,test_probs),3),
       'test_ROC_AUC':round(roc_auc_score(yte,test_probs),3)})
print('Confusion matrix:\n',confusion_matrix(yte,test_probs>=best_t))

**Follow-ups:** How would you select a threshold using the analyst team's daily alert capacity? How would you use stratified evaluation by region/sector/language? How would you estimate precision when labels are missing-not-at-random? What changes when labels have review latency?

**Caveat:** The demonstration uses synthetic arrays rather than real chronological data. In a genuine Bloomberg research setup, perform timestamp-aware splits, entity/event group isolation as appropriate, and test on newly arriving articles.

### Regression, overfitting, PCA, missing data — rapid oral Q&A

**Q: When linear vs gradient-boosted trees?** Linear models are strong transparent baselines; boosting captures nonlinearity/interactions, handles varied tabular signals but may overfit and needs calibration/interpretability.

**Q: L1 vs L2?** L1 can induce sparsity, L2 shrinks coefficients smoothly; scale input features when comparing coefficient penalties.

**Q: Missing data?** First ask *why* missing (MCAR/MAR/MNAR); retain missingness indicators where meaningful, use group-aware imputers fitted only on train, audit coverage bias.

**Q: PCA?** Unsupervised orthogonal projections maximizing variance; can reduce collinearity/dimension but high variance isn't necessarily predictive. Fit PCA on train only; standardize if feature scales differ.

**Q: How test drift?** Monitor input distributions, missingness, segment-level performance, calibration, label delays and business impact; distinguish drift from changes in data pipeline.

## 3. Sustainable finance use case A — Estimating missing corporate GHG emissions

**Research problem:** Many companies do not disclose full emissions. Estimate **Scope 1** (direct operations), **Scope 2** (purchased energy) or **Scope 3** (value-chain emissions) with careful scope separation.

**Features:** revenue, assets, employees, industry, country, energy mix, production output; ensure features known at prediction date.

**Baselines:** sector median intensity × revenue; linear/ridge on log emissions; stronger model: gradient-boosted trees with sector interactions.

**Evaluation:** company-grouped and forward-time holdout, MAE/RMSE on `log1p` emissions and **original units**, sector-conditioned error, bias by size/disclosure status, uncertainty intervals.

**Key research risk:** Training on reporting firms and predicting on non-reporting firms introduces *selection bias*. Furthermore, reported totals can follow inconsistent boundaries/methodologies.

**Production:** preserve source, methodology, vintage, unit, model version, confidence/coverage flags; never silently replace reported values with estimates.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

N=600
industry=rng.choice(['utilities','technology','industrials'],N,p=[.25,.4,.35])
revenue=rng.lognormal(mean=5,sigma=1.0,size=N) # toy $m units
intensity=pd.Series(industry).map({'utilities':8.,'technology':0.35,'industrials':2.0}).to_numpy()
actual_emissions=np.exp(np.log(revenue*intensity+1)+rng.normal(0,.35,N))
df=pd.DataFrame({'industry':industry,'revenue_m':revenue,'emissions_tonnes':actual_emissions})
train_idx,test_idx=train_test_split(np.arange(N),test_size=.25,random_state=42)
train_df,test_df=df.iloc[train_idx].copy(),df.iloc[test_idx].copy()
sector_intensity=(train_df.assign(intensity=train_df.emissions_tonnes/train_df.revenue_m)
                  .groupby('industry').intensity.median())
sector_preds=test_df.revenue_m*test_df.industry.map(sector_intensity)
# Demonstration baseline: a compact feature transform, model fit on train only
features=lambda t: pd.DataFrame({'log_revenue':np.log1p(t.revenue_m),
                                'utilities':(t.industry=='utilities').astype(int),
                                'industrials':(t.industry=='industrials').astype(int)})
reg=HistGradientBoostingRegressor(max_iter=80,max_leaf_nodes=12,l2_regularization=2,random_state=42)
reg.fit(features(train_df),np.log1p(train_df.emissions_tonnes))
model_preds=np.expm1(reg.predict(features(test_df)))
print('Sector intensity baseline MAE:', round(mean_absolute_error(test_df.emissions_tonnes,sector_preds),2))
print('GBM MAE (toy data):',round(mean_absolute_error(test_df.emissions_tonnes,model_preds),2))
print('GBM log MAE:',round(mean_absolute_error(np.log1p(test_df.emissions_tonnes),np.log1p(np.maximum(model_preds,0))),3))

**Interview follow-up:** Our toy data hides the core issue: emissions disclosures are *not random*. Explain how to estimate out-of-domain uncertainty, build sector-specific models where justified, evaluate on independently audited disclosed cases, and communicate unvalidated populations.

**Task:** Add `country` and `year`; perform a genuine year-based split and compare baseline vs model with performance by sector. Discuss feature vintage and restatements.

## 4. Sustainable finance use case B — Real-time news controversy detection

**Pipeline:** news ingestion → language detection → entity linking → de-duplication / story clustering → event and ESG-category detection → severity + factuality → evidence and provenance → client delivery → analyst feedback.

**Research approaches:** TF-IDF + logistic regression baseline; financial-domain transformer encoder fine-tuning; multi-label classifier; LLM-assisted structured event extraction with schema and evidence constraints.

**Important distinctions:** A company **mentioned** in a controversy may be a victim, witness or accused party. Event occurrence is not guilt. Facts and allegations must be kept separate; a company name and negative sentiment are not sufficient.

**Label design:** topic (labor, environment, corruption, safety), entity-role, event time, confidence, confirmed/alleged, severity; adjudicated labeling instructions and inter-annotator agreement.

**Metrics:** entity-link accuracy, event recall, false alerts per issuer-day, time-to-alert, PR-AUC, calibration, latency p50/p95/p99, fairness across language/region, duplicate rate and analyst acceptance.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
toy_texts=[
 'Regulator investigates factory after river pollution allegation',
 'Company announces improved earnings and new contracts',
 'Workers protest over unsafe conditions at the plant',
 'Retail chain opens twenty new stores',
 'Investigators examine bribery allegations involving supplier',
 'Company publishes annual sales outlook',
 'Industrial site fined after waste discharge',
 'Technology firm hires new research director',
 'Safety inspectors report possible violations at warehouse',
 'Bank launches upgraded mobile application',
 'Employee union alleges labor law violations',
 'Company announces quarterly dividend']
toy_y=np.array([1,0,1,0,1,0,1,0,1,0,1,0])
clf=make_pipeline(TfidfVectorizer(ngram_range=(1,2)),LogisticRegression(max_iter=1000))
clf.fit(toy_texts,toy_y)
for sample in ['Regulator reviews potential contamination at plant','Company reports strong quarterly revenue']:
    print(sample, '→',round(float(clf.predict_proba([sample])[0,1]),3))
print('This is a pipeline demo only, NOT a credible benchmark (tiny training set, no held-out data).')

### NLP / LLM interview questions — concise model answers

**Q: Why do transformers outperform bag-of-words in context-dependent classification?** Self-attention learns contextual representations and can capture semantics and relations beyond literal n-grams; it also carries latency/compute costs and may struggle with long documents.

**Q: Encoder vs decoder?** Encoder-style models often suit classification/tagging; decoder-style LLMs are flexible for extraction and reasoning but typically cost more to serve and require constrained outputs, validation, and hallucination controls.

**Q: RAG?** Retrieve documents, rerank evidence, generate grounded responses with citations. RAG may support explanations and knowledge lookup, but core event detection still needs evaluated entity/event extraction and deterministic structured outputs.

**Q: Embeddings?** Dense numerical representations useful for semantic similarity, retrieval and clustering; evaluate domain/language drift and false-nearest-neighbor risk.

**Q: Fine-tuning vs prompting?** Fine-tuning may improve repeatable domain extraction at scale, but needs high-quality labels and maintenance; prompting is fast to iterate, potentially slower/more costly and model-version sensitive.

**Q: Class imbalance?** Prefer PR-centric metrics, thresholding tuned to review capacity, hard-negative mining, cost-sensitive training, and per-segment recall.

**Q: LLM evaluation?** Separate structured validity, fact extraction accuracy, attribution/evidence faithfulness, latency and cost. Human adjudication for critical borderline cases.

## 5. Sustainable finance use case C — Financial materiality

**Key question:** Does an event plausibly affect enterprise value, operational continuity, financing costs or asset risk? This is not identical to asking whether the event is environmentally or socially important.

**Potential features:** incident type, verified magnitude, facility relevance, geography, issuer exposure, fines, production losses, supply-chain dependencies and estimated time to remediation.

**Label traps:** News coverage is selective; large issuers attract more articles; market movements have many confounders; security returns are not a clean causal label for ESG impact.

**Suggested approach:** transparent domain-based scoring baseline, supervised ranking trained on carefully reviewed cases, analyst explanations, and event-study analysis as **supporting evidence** rather than causal proof.

**Research design:** compare to sector/size-matched issuers, avoid outcome leakage, use confidence intervals, control for concurrent earnings/events, inspect delayed market reactions.

**Interview prompt:** 'A firm experiences a chemical spill. Design a score for financial materiality.' Explain assumptions, observable signals, ex-ante timestamps, ground truth, uncertainty, and human escalation.

## 6. Explainability, uncertainty, monitoring and governance

**Global** interpretation: which features broadly drive the model? **Local** explanation: which inputs affected a specific company estimate?

Use interpretable sector-median and linear baselines first; permutation importance for model-level relevance; SHAP for local/global additive attributions (with care under feature correlations). **Feature attribution is not causal explanation.**

**Uncertainty:** bootstrap/ensemble estimates, conformal prediction where assumptions are tenable, interval coverage by sector/size, population-shift diagnostics.

**Controls:** reproducible feature snapshots, immutable input/provenance, model registry/version, reproducible evaluations, human overrides, audit logs, explicit fallback and rollback.

**Monitoring:** data freshness, missingness, embedding drift, segment drift, model calibration, label-based evaluation after delay, p95 latency, alert backlog, false-positive review rates.

**Escalation:** poorly supported, very severe events should typically be reviewed rather than presented as verified facts.

## 7. System design — sketch on whiteboard

```text
NEWS FEEDS -> INGEST / TIMESTAMP -> NORMALISE -> ENTITY LINK -> DEDUPE
                                                            |
                                                            v
                                         NLP CLASSIFIER / EVENT EXTRACTION
                                                            |
                                           VALIDATE FACTS + PROVENANCE
                                                            |
                                SCORE SEVERITY / MATERIALITY + CALIBRATE
                                                            |
                                        ALERT API + REVIEW WORK QUEUE
                                                            |
                                 ANNOTATION -> EVALUATION -> TRAINING

Cross-cutting: observability | model registry | feature/data versioning |
DLQ/retry/idempotency | auditing | rollback | access control
```

**Discuss:** immutable event IDs, out-of-order news, exactly-once *business effect* via idempotent writes, batch vs stream, throughput and latency targets, vector database's role if retrieval is needed, adversarial/poor-quality sources, schema migration, and model shadow deployment.

**Design challenge:** How do you backfill a missed news feed without sending duplicate alerts? Which predictions do you reprocess after an entity-linking model upgrade?

## 8. Behavioural and research-project stories

Prepare **three 2-minute STAR stories**, explicitly stating **your contribution**, quantitative outcomes only where known, trade-offs and lessons. Possible relevant themes from your experience: ESG research/model pipelines; anomaly detection in market data; quant ML pipeline deployment; moving experimental AI ideas toward controlled production.

**Question bank:**
- Walk me through one ML research idea from hypothesis to production, including rejected alternatives.
- Tell us about low-quality labels. How did you assess inter-annotator disagreement?
- What happened when model performance deteriorated?
- Describe a disagreement with domain experts over an evaluation metric.
- Explain why you prefer research engineering rather than conventional backend development.
- What would you change in your past ESG modelling work today?

**Why Bloomberg?** Tie your interest to ML models that convert unstructured real-time financial information into traceable, decision-relevant analytics; describe the attraction of research plus production at scale, not just the brand.

## 9. Mock interview — 60 minutes

**0–8 min:** 'Tell me about your background, strongest ML project, and why Sustainable Finance.'

**8–25 min:** Implement Exercise 1 on a blank editor. Follow-up: unsorted input? streaming events? bounded-lateness event times? per-issuer windows?

**25–40 min:** Design emissions estimation under selective reporting. Define validation and model uncertainty.

**40–52 min:** Design NLP controversy alerts; discuss precision/recall, entity-role, labels, production constraints.

**52–60 min:** Questions to interviewer.

**Self-score (0–2 each):** clarified requirements; baseline; sound algorithm/ML choice; leakage-safe evaluation; monitoring; communicated trade-offs; wrote tested code; addressed ambiguity. Total /16.

**Good questions to ask:** How much of the role focuses on novel modelling versus serving systems? What are the bottlenecks—annotation, entity linking, calibration or latency? How are research proposals compared and productionized? How is sustainability financial-materiality ground truth established?

## 10. Additional practice problems (try without looking anything up)

**Algorithms:**
1. Maintain top 3 issuers by weighted severity across an unbounded stream (heap + counter; update complications).
2. Given time-stamped events, merge consecutive events separated by at most 15 minutes into incident clusters.
3. Given two article lists, return the Jaccard similarity of normalized entity sets.
4. Find shortest supplier path between firms with BFS.
5. Implement an LRU cache for issuer metadata (`OrderedDict` or doubly linked list + hash map).

**ML research:**
6. Design a fair evaluation when one issuer has 1,000 articles and another has 2.
7. How would you estimate a confidence interval around a model's test MAE? Discuss correlated samples.
8. Why might a high AUC model be poor at a low daily-alert budget?
9. Compare a sector median baseline to a boosted tree for missing Scope 3 emissions.
10. Distinguish model drift from a news-provider taxonomy change.

**System design:**
11. Create idempotent alert delivery after retries.
12. Plan an A/B or shadow test for a redesigned entity-linking system.
13. Build an annotation workflow for disagreement over allegations vs confirmed facts.
14. Design a lineage record explaining a company's ESG estimate to a client.
15. Decide whether the pipeline should use a vector DB, a feature store, both, or neither.

## 11. Research discussion prompts: deeper model answers

**Q: 'How would you verify real-time model gains?'** Define a frozen time cutoff and label policy. Hold out event clusters or entity/time slices to limit duplicate leakage. Compare a strong baseline to challenger on PR-AUC, severe-event recall at fixed review budget, entity attribution and p95 latency. Run shadow inference, sample errors, track drift and then deploy under control gates.

**Q: 'How do you handle label scarcity?'** Start with adjudicated high-quality gold labels, targeted active learning, weak supervision cautiously, semi-supervised techniques only with leakage-safe checks, hard negatives from near-miss entity roles, and a continuously refreshed stratified test set.

**Q: 'What makes a good research experiment?'** Explicit hypothesis; fixed dataset/split and baseline; well-defined metric; repeatable seeds and environment; confidence intervals; ablations; error analysis; inference cost and latency; decision criterion before seeing test results.

**Q: 'When would you reject an LLM?'** When a small classifier meets recall and latency goals at significantly lower cost and greater reproducibility; when results are not grounded enough for customer-facing attribution; or where deterministic rules and audited datasets work better.

**Q: 'How do you prevent target leakage in financial time series?'** Join on **availability/as-of time**, not just the event's nominal accounting date. Lag revised fundamentals appropriately, isolate restatements, and run rolling-forward evaluation.

## 12. Sources and scope

- [Bloomberg Senior ML Research Engineer — Sustainable Finance, job description](https://uk.indeed.com/viewjob?jk=e13388d292c898f2) (role scope)
- [Bloomberg ML Research Engineer candidate interview report](https://leetcode.com/discuss/post/2070046/bloomberg-phoneonsite-interview-machine-learning-research-engineer/) (anecdotal, different team)
- [Bloomberg ML interview discussion](https://www.glassdoor.co.uk/Interview/Bloomberg-Machine-Learning-Engineer-Interview-Questions-EI_IE3096.0%2C9_KO10%2C35.htm) (candidate-reported; format varies)

This notebook is a preparation resource, not Bloomberg's official question bank. Synthetic examples are educational; they must not be interpreted as investable ESG analytics.

**Dependencies:** `numpy`, `pandas`, `scikit-learn`. Installation if needed: `%pip install numpy pandas scikit-learn`. No statsmodels required.